In [1]:
# Importing the libraries
import pandas as pd
import numpy as np
from utils.fake_na_detection_and_cleaning import FakeNullDetector
from utils.sql_connector import SQLConnector
from utils.data_type_converter import DataTypeConverter
from utils.normalizer import Normalizer
from utils.mapping_categorical import MappingCategorical

In [2]:
# loading of 2022 datasets and object creations
db=SQLConnector('Stack_Overflow_Survey')
db.connect()
query='select * from Bronze.Survey_2022'
Survey_2022_df = db.read_query(query)

FakeNullDetector_obj = FakeNullDetector()
DataTypeConverter_obj = DataTypeConverter()
Normalizer_obj = Normalizer()
MappingCategorical_obj = MappingCategorical()

Successfully Connected to Stack_Overflow_Survey
Query executed successfully


c:\Users\Ayush\Git Repo\Stack-Over-Flow-Survey-Data-Engineering-Project\Data Warehouse\Silver Layer\utils\sql_connector.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, self.conn)


In [3]:
# Null detector and remover 
Survey_2022_df_cleaned=Survey_2022_df.copy()
FakeNullDetector_obj.detect_fake_nulls(Survey_2022_df_cleaned)
FakeNullDetector_obj.replace_fake_nulls(Survey_2022_df_cleaned)

{'Accessibility': {'NA': np.int64(6024)},
 'Age': {'NA': np.int64(2322)},
 'Blockchain': {'NA': np.int64(2197)},
 'BuyNewTool': {'NA': np.int64(5305)},
 'CodingActivities': {'NA': np.int64(14369)},
 'CompFreq': {'NA': np.int64(28843)},
 'CompTotal': {'NA': np.int64(34846)},
 'ConvertedCompYearly': {'NA': np.int64(35197)},
 'Country': {'NA': np.int64(1497)},
 'Currency': {'NA': np.int64(22004)},
 'DatabaseHaveWorkedWith': {'NA': np.int64(13147)},
 'DatabaseWantToWorkWith': {'NA': np.int64(22254)},
 'DevType': {'NA': np.int64(11966)},
 'EdLevel': {'NA': np.int64(1697)},
 'Employment': {'NA': np.int64(1559)},
 'Ethnicity': {'NA': np.int64(3794)},
 'Frequency_1': {'NA': np.int64(37897)},
 'Frequency_2': {'NA': np.int64(37924)},
 'Frequency_3': {'NA': np.int64(38753)},
 'Gender': {'NA': np.int64(2415)},
 'ICorPM': {'NA': np.int64(36985)},
 'Knowledge_1': {'NA': np.int64(37464)},
 'Knowledge_2': {'NA': np.int64(38295)},
 'Knowledge_3': {'NA': np.int64(38135)},
 'Knowledge_4': {'NA': np.int64

In [4]:
Survey_2022_df_cleaned['Employment'].value_counts()

In [5]:
# Mormalization of categorical columns : Basic mapping of values to reduce the number of unique values in each column and make it more consistent for analysis and visualization.
un_normalized_cols_name = ['Employment', 'EdLevel', 'Age', 'OpSysPersonal use', 'OpSysProfessional use', 'OrgSize', 'Trans', 'SOVisitFreq', 'SOAccount', 'SOPartFreq', 'SOComm', 'NEWSOSites', 'SurveyLength', 'SurveyEase', 'MainBranch']
normalized_cols_name = ['Employment', 'Education_Level', 'Age', 'OperatingSystem_Personal', 'OperatingSystem_Professional', 'Organization_Size', 'TransGender', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participation_Frequency', 'StackOverflow_Community_Experience', 'NewStackOverflow_Sites', 'Survey_Length', 'Survey_Ease', 'Current_Profession']
columns_map = [
    MappingCategorical_obj.get_map('employment_map'),
    MappingCategorical_obj.get_map('ed_level_map'),
    MappingCategorical_obj.get_map('age_map'),
    MappingCategorical_obj.get_map('operating_system_map'),
    MappingCategorical_obj.get_map('operating_system_map'),
    MappingCategorical_obj.get_map('org_mapping'),
    MappingCategorical_obj.get_map('trans_map'),
    MappingCategorical_obj.get_map('visit_freq_map'),
    MappingCategorical_obj.get_map('so_account_map'),
    MappingCategorical_obj.get_map('part_freq_map'),
    MappingCategorical_obj.get_map('comm_map'),
    MappingCategorical_obj.get_map('new_so_sites_map'),
    MappingCategorical_obj.get_map('survey_length_map'),
    MappingCategorical_obj.get_map('survey_ease_map'),
    MappingCategorical_obj.get_map('main_branch_map')
]
Survey_2022_df_cleaned = Normalizer_obj.normalize_categorical_columns_manual_mapping(Survey_2022_df_cleaned, un_normalized_cols_name, normalized_cols_name, columns_map)

[nan 'Employed, full-time' 'Student, full-time' 'Student, part-time'
 'Not employed, but looking for work'
 'Independent contractor, freelancer, or self-employed'
 'Employed, full-time;Independent contractor, freelancer, or self-employed'
 'Employed, part-time' 'Student, part-time;Employed, part-time'
 'Not employed, and not looking for work'
 'Student, full-time;Employed, part-time'
 'Employed, full-time;Student, part-time'
 'Employed, full-time;Student, full-time'
 'Student, part-time;Independent contractor, freelancer, or self-employed'
 'Retired' 'Student, full-time;Not employed, but looking for work'
 'I prefer not to say'
 'Student, full-time;Independent contractor, freelancer, or self-employed'
 'Student, full-time;Not employed, and not looking for work'
 'Not employed, but looking for work;Independent contractor, freelancer, or self-employed'
 'Employed, full-time;Student, part-time;Independent contractor, freelancer, or self-employed'
 'Independent contractor, freelancer, or s

In [6]:
# Mormalization of categorical columns : changing NA to more meaningful values and make it more consistent for analysis and visualization.
nan_replacer_columns=['CompFreq']
cleaned_nan_replacer_columns=['Compensation_Frequency']
if nan_replacer_columns:
    Survey_2022_df_cleaned=Normalizer_obj.normalize_na_replacer_columns(Survey_2022_df_cleaned,nan_replacer_columns, cleaned_nan_replacer_columns)

In [7]:
# Mormalization of categorical columns : Multi-select columns where respondents could select multiple options, resulting in semicolon-separated values.
multi_select_cols = ['Gender', 'Sexuality', 'Ethnicity', 'Accessibility', 'MentalHealth']
multi_select_normalized = ['Gender_Clean', 'Sexuality_Clean', 'Ethnicity_Clean', 'Accessibility_Status', 'Mental_Health_Status']
multi_select_maps = [
    MappingCategorical_obj.get_map('gender_map'),
    MappingCategorical_obj.get_map('sexuality_map'),
    MappingCategorical_obj.get_map('ethnicity_map'),
    MappingCategorical_obj.get_map('accessibility_map'),
    MappingCategorical_obj.get_map('mental_health_map')
]
if multi_select_cols:
    Normalizer_obj.normalize_categorical_columns_non_exploding(Survey_2022_df_cleaned, multi_select_cols, multi_select_normalized, multi_select_maps)

Gender_Clean
Man                   64607
Unknown                3587
Woman                  3399
Non-binary / GNC        983
Diverse / Multiple      692
Name: count, dtype: int64
Sexuality_Clean
Straight              55238
Unknown               11053
Bisexual               2700
Diverse / Multiple     1422
Gay or Lesbian         1382
Self-described         1079
Queer                   394
Name: count, dtype: int64
Ethnicity_Clean
Unknown                 48677
Diverse / Multiple      19819
Middle Eastern           1540
South Asian               803
Self-described            798
Southeast Asian           792
East Asian                500
Multiracial/Biracial      339
Name: count, dtype: int64
Accessibility_Status
None                           63064
Unknown                         7657
Visual Impairment                981
Self-described                   579
Hearing Impairment               436
Diverse / Multiple               234
Mobility (Walking/Standing)      186
Mobility (Typing)    

In [8]:
# Mormalization of categorical columns : we will create a mapping to group similar roles and responses together, reducing the number of unique values while preserving the overall meaning.
tech_stack_cols = ['LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'MiscTechHaveWorkedWith', 'MiscTechWantToWorkWith', 'ToolsTechHaveWorkedWith', 'ToolsTechWantToWorkWith', 'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'OfficeStackSyncHaveWorkedWith', 'OfficeStackSyncWantToWorkWith']
manual_mapping_cols = ['DevType', 'LearnCode']
all_target_cols = manual_mapping_cols + tech_stack_cols
manual_maps = {
    'DevType': MappingCategorical_obj.get_map('dev_type_map'),
    'LearnCode': MappingCategorical_obj.get_map('learn_code_map')
}
bridge_results = {}
target_names = [col + '_Clean' for col in all_target_cols]
maps_to_use = [manual_maps.get(col) for col in all_target_cols]
bridge_results = Normalizer_obj.normalize_categorical_columns_exploding(
    Survey_2022_df_cleaned, 
    all_target_cols, 
    target_names, 
    maps_to_use
)

--- Distribution for DevType_Clean ---
DevType_Clean
Full-stack             28701
Back-end               26595
Front-end              15915
Other/Unknown          11966
DevOps                 11453
Desktop/Enterprise      9546
Mobile                  7634
Product Manager         6411
Student                 5595
DBA                     4934
SysAdmin                4908
Embedded/IoT            3923
Other                   3920
SRE                     3875
Designer                3764
Data Engineer           3600
Engineering Manager     3574
Data Scientist/ML       3424
Data/BI Analyst         3201
QA/Testing              3096
Researcher              2709
Educator                2090
Game/Graphics           1837
Executive               1805
Scientist               1762
Marketing/Sales          518
Name: count, dtype: int64
------------------------------
--- Distribution for LearnCode_Clean ---
LearnCode_Clean
Other/Unknown            130742
Physical Media            38994
Online Certific

In [9]:
# Cleaning the year Columns
experience_cols = ['YearsCode', 'YearsCodePro']
Survey_2022_df_cleaned = Normalizer_obj.clean_years_columns(Survey_2022_df_cleaned, experience_cols)

In [10]:
# Cleaning the Currency column and Salary Columns
col = 'Currency'
Survey_2022_df_cleaned[col] = Survey_2022_df_cleaned[col].str.split().str[0]
nan_replacer_cols = ['Currency']
cleaned_nan_cols = ['Currency_Code']
Survey_2022_df_cleaned = Normalizer_obj.normalize_na_replacer_columns(
    Survey_2022_df_cleaned,
    nan_replacer_cols, 
    cleaned_nan_cols,
    replacer_value="Not Available"
)
numeric_target_cols = ['CompTotal', 'ConvertedCompYearly']
Survey_2022_df_cleaned = DataTypeConverter_obj.string_to_numeric(Survey_2022_df_cleaned, numeric_target_cols)
Normalizer_obj.fill_na_and_remove_outlier_percentile_method(Survey_2022_df_cleaned, 'ConvertedCompYearly', 0.01, 0.95)
Normalizer_obj.fill_na_and_remove_outlier_percentile_method(Survey_2022_df_cleaned, 'CompTotal', 0.01, 0.95)

In [11]:
# Dropping the unrequired columns
# 1. Original columns that now have "Clean" versions
raw_redundant_cols = [col for col in ['Employment', 'EdLevel', 'Age', 'OpSysPersonal use', 'OpSysProfessional use', 'OrgSize', 'Trans', 'SOVisitFreq', 'SOAccount', 'SOPartFreq', 'SOComm', 'NEWSOSites', 'SurveyLength', 'SurveyEase', 'MainBranch'] if col != 'MainBranch'] + ['CompFreq'] + ['Gender', 'Sexuality', 'Ethnicity', 'Accessibility', 'MentalHealth'] + ['Currency']
# 2. Raw multi-select strings (already exploded into bridge_results)
multi_select_strings = all_target_cols
total_drop_list = list(set(raw_redundant_cols + multi_select_strings))
Survey_2022_df_cleaned.drop(columns=total_drop_list, inplace=True, errors='ignore')
print(f"Final Column Count: {len(Survey_2022_df_cleaned.columns)}")
print(Survey_2022_df_cleaned.columns.tolist())

Final Column Count: 59
['ResponseId', 'MainBranch', 'RemoteWork', 'CodingActivities', 'LearnCodeOnline', 'LearnCodeCoursesCert', 'YearsCode', 'YearsCodePro', 'PurchaseInfluence', 'BuyNewTool', 'Country', 'CompTotal', 'VersionControlSystem', 'VCInteraction', 'VCHostingPersonal use', 'VCHostingProfessional use', 'Blockchain', 'TBranch', 'ICorPM', 'WorkExp', 'Knowledge_1', 'Knowledge_2', 'Knowledge_3', 'Knowledge_4', 'Knowledge_5', 'Knowledge_6', 'Knowledge_7', 'Frequency_1', 'Frequency_2', 'Frequency_3', 'TimeSearching', 'TimeAnswering', 'Onboarding', 'ProfessionalTech', 'TrueFalse_1', 'TrueFalse_2', 'TrueFalse_3', 'ConvertedCompYearly', 'SurveyYear', 'Education_Level', 'OperatingSystem_Personal', 'OperatingSystem_Professional', 'Organization_Size', 'TransGender', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participation_Frequency', 'StackOverflow_Community_Experience', 'NewStackOverflow_Sites', 'Survey_Length', 'Survey_Ease', 'Current_Profession', 'Co

In [12]:
# Data Type Conversion
numeric_cols = ['YearsCode', 'YearsCodePro', 'CompTotal', 'ConvertedCompYearly', 'SurveyYear']
categorical_cols = ['MainBranch', 'Country', 'Education_Level', 'OperatingSystem_Personal', 'OperatingSystem_Professional', 'Organization_Size', 'TransGender', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participation_Frequency', 'StackOverflow_Community_Experience', 'NewStackOverflow_Sites', 'Survey_Length', 'Survey_Ease', 'Compensation_Frequency', 'Gender_Clean', 'Sexuality_Clean', 'Ethnicity_Clean', 'Accessibility_Status', 'Mental_Health_Status', 'Current_Profession', 'Currency_Code']
DataTypeConverter_obj.string_to_category(Survey_2022_df_cleaned, categorical_cols)
DataTypeConverter_obj.string_to_numeric(Survey_2022_df_cleaned, numeric_cols)
Survey_2022_df_cleaned['SurveyYear'] = Survey_2022_df_cleaned['SurveyYear'].fillna(0).astype('datetime64[ns]')
print(Survey_2022_df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73268 entries, 0 to 73267
Data columns (total 59 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   ResponseId                             73268 non-null  object        
 1   MainBranch                             73268 non-null  category      
 2   RemoteWork                             58958 non-null  object        
 3   CodingActivities                       58899 non-null  object        
 4   LearnCodeOnline                        50685 non-null  object        
 5   LearnCodeCoursesCert                   29389 non-null  object        
 6   YearsCode                              73268 non-null  int64         
 7   YearsCodePro                           73268 non-null  int64         
 8   PurchaseInfluence                      50969 non-null  object        
 9   BuyNewTool                             67963 non-null  object

In [13]:
for table_name, bridge_df in bridge_results.items():
    value_col = [col for col in bridge_df.columns if col != 'ResponseId'][0]
    bridge_results[table_name][value_col] = bridge_df[value_col].astype('category')
    bridge_results[table_name] = DataTypeConverter_obj.string_to_numeric(bridge_results[table_name], ['ResponseId'])
    print(f"Fixed types for {table_name}: {bridge_results[table_name][value_col].dtype}")

Fixed types for DevType_Clean: category
Fixed types for LearnCode_Clean: category
Fixed types for LanguageHaveWorkedWith_Clean: category
Fixed types for LanguageWantToWorkWith_Clean: category
Fixed types for DatabaseHaveWorkedWith_Clean: category
Fixed types for DatabaseWantToWorkWith_Clean: category
Fixed types for PlatformHaveWorkedWith_Clean: category
Fixed types for PlatformWantToWorkWith_Clean: category
Fixed types for WebframeHaveWorkedWith_Clean: category
Fixed types for WebframeWantToWorkWith_Clean: category
Fixed types for MiscTechHaveWorkedWith_Clean: category
Fixed types for MiscTechWantToWorkWith_Clean: category
Fixed types for ToolsTechHaveWorkedWith_Clean: category
Fixed types for ToolsTechWantToWorkWith_Clean: category
Fixed types for NEWCollabToolsHaveWorkedWith_Clean: category
Fixed types for NEWCollabToolsWantToWorkWith_Clean: category
Fixed types for OfficeStackAsyncHaveWorkedWith_Clean: category
Fixed types for OfficeStackAsyncWantToWorkWith_Clean: category
Fixed ty

In [14]:
# Writing back to SQL
# Central Fact Table 2022
db.write_to_sql(df=Survey_2022_df_cleaned, schema='Silver', table_name='Survey_2022')
# Bridge Tables for Tech Stack and Manual Mapping Columns
for table_name, bridge_df in bridge_results.items():
    db.write_to_sql(df=bridge_df, schema='Silver', table_name=f"Bridge_{table_name}_2022")
db.close()

DataFrame written to Silver.Survey_2022 successfully.
DataFrame written to Silver.Bridge_DevType_Clean_2022 successfully.
DataFrame written to Silver.Bridge_LearnCode_Clean_2022 successfully.
DataFrame written to Silver.Bridge_LanguageHaveWorkedWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_LanguageWantToWorkWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_DatabaseHaveWorkedWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_DatabaseWantToWorkWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_PlatformHaveWorkedWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_PlatformWantToWorkWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_WebframeHaveWorkedWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_WebframeWantToWorkWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_MiscTechHaveWorkedWith_Clean_2022 successfully.
DataFrame written to Silver.Bridge_MiscTechWantToWorkWith_Cle

In [15]:
db.close()